# JackSparrow — Delta Exchange India (v43 training)

Train a **v43-compatible** bundle in **Google Colab** (or local Jupyter), download the export folder, copy into `agent/model_storage/<your_bundle>/`, and set `MODEL_DIR` / `AGENT_MODEL_DIR`.

**Parity**: `metadata_v43.json` `features` must match `feature_store/jacksparrow_v43_contract.V43_CANONICAL_FEATURES` (order-sensitive). This notebook imports `JackSparrowV43FeatureEngineer` from the repo.

**Credentials**: use the same `DELTA_EXCHANGE_API_KEY` / `DELTA_EXCHANGE_API_SECRET` as the agent. In Colab, store them as **Secrets**. Never commit keys.

**Placeholder model**: the default export uses `SklearnRegressorV43Export` wrapping `HistGradientBoostingRegressor` so `joblib` loads in the agent from the `feature_store` module path. Replace the training cell with your real ensemble before production.


## 1. Repository on `sys.path`

- **Colab**: clone your fork (set `JACKSPARROW_REPO_URL`) or upload this project and set `JACKSPARROW_REPO_ROOT`.
- **Local** (repo root as cwd): this cell adds the current directory automatically.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_ROOT = Path(os.environ.get("JACKSPARROW_REPO_ROOT", "")).resolve()
if not REPO_ROOT.is_dir() or not (REPO_ROOT / "feature_store").is_dir():
    cwd = Path.cwd().resolve()
    if (cwd / "feature_store").is_dir():
        REPO_ROOT = cwd
    else:
        url = os.environ.get("JACKSPARROW_REPO_URL", "").strip()
        branch = os.environ.get("JACKSPARROW_REPO_BRANCH", "main").strip()
        clone_to = Path(os.environ.get("JACKSPARROW_CLONE_DIR", "/content/JackSparrow"))
        if not url:
            raise RuntimeError(
                "Could not find feature_store/. Set JACKSPARROW_REPO_ROOT to repo root, "
                "run from repo root, or set JACKSPARROW_REPO_URL for git clone."
            )
        if not clone_to.is_dir():
            subprocess.run(
                ["git", "clone", "--depth", "1", "-b", branch, url, str(clone_to)],
                check=True,
            )
        REPO_ROOT = clone_to.resolve()

sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)


## 2. Dependencies (align with `agent/requirements.txt`)


In [ ]:
%pip install -q "numpy>=1.24" "pandas>=2.0" "httpx>=0.27" "joblib>=1.3" "scikit-learn>=1.3" "xgboost==2.0.2" "lightgbm==4.1.0"
import numpy as np
import pandas as pd
import httpx
import hmac
import hashlib
import time
import json
from pathlib import Path
from urllib.parse import urlencode
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

print("deps ok", np.__version__)


## 3. Delta India API keys

Colab: **Secrets** → `DELTA_EXCHANGE_API_KEY`, `DELTA_EXCHANGE_API_SECRET`. Local: export env vars or edit the two lines below (do not commit).


In [ ]:
def _load_secret(name: str, default: str = "") -> str:
    v = os.environ.get(name, "").strip()
    if v:
        return v
    try:
        from google.colab import userdata  # type: ignore

        return userdata.get(name)
    except Exception:
        return default


API_KEY = _load_secret("DELTA_EXCHANGE_API_KEY", "")
API_SECRET = _load_secret("DELTA_EXCHANGE_API_SECRET", "")
BASE_URL = os.environ.get("DELTA_EXCHANGE_BASE_URL", "https://api.india.delta.exchange").rstrip("/")

if not API_KEY or not API_SECRET:
    raise RuntimeError("Missing Delta API credentials")

print("BASE_URL =", BASE_URL)


## 4. Paginated candle download (signed GET)

Calls `GET /v2/history/candles` with HMAC headers (same layout as `agent/data/delta_client.py`).


In [ ]:
def delta_signed_get(path: str, params: Dict[str, Any]) -> Dict[str, Any]:
    ts = str(int(time.time()))
    qs = "?" + urlencode(params) if params else ""
    msg = f"GET{ts}{path}{qs}"
    sig = hmac.new(API_SECRET.encode("utf-8"), msg.encode("utf-8"), hashlib.sha256).hexdigest()
    headers = {
        "api-key": API_KEY,
        "timestamp": ts,
        "signature": sig,
        "Content-Type": "application/json",
        "User-Agent": "JackSparrow-Colab-Training/1.0",
        "recv-window": "5000",
    }
    url = f"{BASE_URL}{path}{qs}"
    with httpx.Client(timeout=60.0) as client:
        r = client.get(url, headers=headers)
        r.raise_for_status()
        return r.json()


def parse_candles(resp: Any) -> List[dict]:
    rows: List[dict] = []
    if isinstance(resp, dict):
        result = resp.get("result")
        if isinstance(result, dict):
            rows = result.get("candles") or []
        elif isinstance(result, list):
            rows = result
    elif isinstance(resp, list):
        rows = resp
    out: List[dict] = []
    for c in rows:
        if not isinstance(c, dict):
            continue
        ts = c.get("time")
        if ts is None:
            continue
        out.append(
            {
                "timestamp": int(ts),
                "open": float(c.get("open", 0) or 0),
                "high": float(c.get("high", 0) or 0),
                "low": float(c.get("low", 0) or 0),
                "close": float(c.get("close", 0) or 0),
                "volume": float(c.get("volume", 0) or 0),
            }
        )
    return out


def fetch_candles_range(symbol: str, resolution: str, end_ts: int, start_ts: int) -> pd.DataFrame:
    params = {
        "symbol": symbol,
        "resolution": resolution.lower(),
        "start": start_ts,
        "end": end_ts,
    }
    raw = delta_signed_get("/v2/history/candles", params)
    rows = parse_candles(raw)
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)
    return df.sort_values("timestamp").reset_index(drop=True)


def fetch_candles_paginated(
    symbol: str,
    resolution: str,
    *,
    target_bars: int = 8000,
    chunk_seconds: Optional[int] = None,
) -> pd.DataFrame:
    res_sec = {"1m": 60, "3m": 180, "5m": 300, "15m": 900, "30m": 1800, "1h": 3600}.get(
        resolution.lower(), 300
    )
    if chunk_seconds is None:
        chunk_seconds = min(2000 * res_sec, 30 * 24 * 3600)
    end_ts = int(datetime.now(timezone.utc).timestamp())
    start_ts = end_ts - chunk_seconds
    parts: List[pd.DataFrame] = []
    total = 0
    pages = 0
    max_pages = 500
    while total < target_bars and pages < max_pages:
        df = fetch_candles_range(symbol, resolution, end_ts, start_ts)
        if df.empty:
            break
        parts.append(df)
        total += len(df)
        end_ts = start_ts - 1
        start_ts = end_ts - chunk_seconds
        pages += 1
        time.sleep(0.12)
    if not parts:
        return pd.DataFrame()
    out = pd.concat(parts, ignore_index=True)
    return out.sort_values("timestamp").drop_duplicates(subset=["timestamp"]).reset_index(drop=True)


SYMBOL = os.environ.get("DELTA_SYMBOL", "BTCUSD")

df5 = fetch_candles_paginated(SYMBOL, "5m", target_bars=6000)
df15 = fetch_candles_paginated(SYMBOL, "15m", target_bars=4000)
df1h = fetch_candles_paginated(SYMBOL, "1h", target_bars=2000)
print("rows 5m/15m/1h", len(df5), len(df15), len(df1h))
df5.head()


## 5. Funding (`FUNDING:SYMBOL` hourly) or zeros


In [ ]:
fund_symbol = f"FUNDING:{SYMBOL}"
try:
    df_fund = fetch_candles_paginated(fund_symbol, "1h", target_bars=min(2000, max(len(df1h), 100)))
    if not df_fund.empty and "close" in df_fund.columns:
        df_fund = df_fund.rename(columns={"close": "funding_rate"})
    else:
        df_fund = pd.DataFrame()
except Exception as exc:
    print("funding skipped:", exc)
    df_fund = pd.DataFrame()

if df_fund.empty and not df1h.empty:
    df_fund = pd.DataFrame({"timestamp": df1h["timestamp"].values, "funding_rate": 0.0})
print("funding rows", len(df_fund))


## 6. Features + target (`JackSparrowV43FeatureEngineer`)


In [ ]:
from feature_store.jacksparrow_v43_feature_engineer import JackSparrowV43FeatureEngineer
from feature_store.jacksparrow_v43_contract import (
    V43_CANONICAL_FEATURES,
    V43_COMPATIBLE_FEATURE_VERSION,
    V43_FORWARD_TARGET_BARS,
)

fe = JackSparrowV43FeatureEngineer()
df_feat = fe.transform(df5, df15, df1h, df_fund, include_target=True)
df_feat = df_feat.dropna(subset=["target"]).reset_index(drop=True)
print(df_feat.shape, "horizon", V43_FORWARD_TARGET_BARS)


## 7. Train (replace with your ensemble)

Default: `HistGradientBoostingRegressor` wrapped in `SklearnRegressorV43Export` (picklable module path). Calibrate `dynamic_threshold` from OOF or validation predictions.


In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from feature_store.v43_sklearn_export_ensemble import SklearnRegressorV43Export

feat_cols = list(V43_CANONICAL_FEATURES)
X = df_feat[feat_cols].astype(float).values
y = df_feat["target"].astype(float).values

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.15, shuffle=False)
reg = HistGradientBoostingRegressor(max_depth=8, learning_rate=0.05, max_iter=200, random_state=42)
reg.fit(X_train, y_train)
pred_val = reg.predict(X_val)
print("val MAE", float(np.mean(np.abs(pred_val - y_val))))

ensemble = SklearnRegressorV43Export(reg)
ensemble.dynamic_threshold = float(np.percentile(reg.predict(X_train), 75))
ensemble.threshold = ensemble.dynamic_threshold


## 8. Export bundle

Download `EXPORT_DIR` from Colab. Copy `metadata_v43.json` + `model_artifact_v43.pkl` into your bundle folder; point `MODEL_DIR` there; restart agent; run `pytest tests/unit/test_jacksparrow_v43_contract.py -q` and `python scripts/smoke_test_v43.py`.


In [ ]:
import joblib

EXPORT_DIR = Path(os.environ.get("JACKSPARROW_EXPORT_DIR", "/content/jacksparrow_v43_export"))
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

artifact = {
    "model": ensemble,
    "feature_engineer": fe,
    "features": list(V43_CANONICAL_FEATURES),
}
joblib.dump(artifact, EXPORT_DIR / "model_artifact_v43.pkl")

agent_load_lines = [
    "artifact = joblib.load(MODEL_ARTIFACT_PATH)",
    'model    = artifact["model"]',
    'features = artifact["features"]',
    'fe       = artifact["feature_engineer"]',
    "df_feat  = fe.transform(df5, df15, df1h, df_funding, include_target=False)",
    'assert len(df_feat) >= 2, "Need at least 2 rows to use the last closed bar"',
    "X        = df_feat[features].iloc[[-2]].values",
    "X_df     = df_feat[features].iloc[[-2]]",
    "proba    = model.predict(X, X_df=X_df)",
]
meta = {
    "version": "v43",
    "compatible_feature_version": V43_COMPATIBLE_FEATURE_VERSION,
    "symbol": SYMBOL,
    "model_name": f"jacksparrow_v43_{SYMBOL}",
    "exported_at": datetime.now(timezone.utc).isoformat(),
    "feature_count": len(V43_CANONICAL_FEATURES),
    "features": list(V43_CANONICAL_FEATURES),
    "training_forward_bars": int(V43_FORWARD_TARGET_BARS),
    "agent_load": "\n".join(agent_load_lines),
}
(EXPORT_DIR / "metadata_v43.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
print("Wrote", EXPORT_DIR)
list(EXPORT_DIR.iterdir())
